In [1]:
%load_ext autoreload
%autoreload 2

import sys, json, os, time, random, datetime
sys.path.append("..")
from functools import partial
import pandas as pd
from dotenv import load_dotenv
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker
from src.abstain import load_thresholds
from src.llm import complete
from src.pipeline import review_contract
from src.eval_llm import score_items, score_absent

load_dotenv()
SPLIT = "dev"      # "test" ONLY for the one final run
LABELS = "../eval/label_set.csv" if SPLIT == "dev" else "../eval/label_set_test.csv"

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
by_title = {c["title"]: c for c in contracts}
chunker = partial(clause_chunks, max_size=1500)
dense = make_dense_ranker(chunker, tag="clause-200-1500-200")
bm25 = make_bm25_ranker(chunker)
hybrid = make_hybrid_ranker([dense, bm25], weights=[1.0, 2.0])
signals = {"bm25": bm25}
thresholds = load_thresholds()


class CountingLLM:
    def __init__(self, fn):
        self.fn, self.calls = fn, 0

    def __call__(self, system, user):
        self.calls += 1
        return self.fn(system, user)


print("ready", SPLIT)

ready dev


In [13]:
OUT = f"../eval/llm_items_{SPLIT}.jsonl"
MAX_NEW = 20        # trial run. Set to None for the full run.

done = set()
if os.path.exists(OUT):
    done = {json.loads(l)["id"] for l in open(OUT, encoding="utf-8")}

items = pd.read_csv(LABELS, dtype=str, keep_default_na=False, encoding="utf-8-sig")
items = items[items["risk"].isin(["Low", "Medium", "High"])]
print(len(items), "labeled items,", len(done), "already done")

new = 0
for _, it in items.iterrows():
    if it["id"] in done:
        continue
    if MAX_NEW is not None and new >= MAX_NEW:
        break
    c = by_title[it["title"]]
    cust = it["customer"].strip()
    customer = None if cust.lower() in ("", "unclear") else cust
    llm = CountingLLM(complete)
    try:
        r = review_contract(c, hybrid, signals, thresholds, llm,
                            clauses=[it["clause"]], customer=customer)[0]
    except Exception as e:
        print("stopped:", type(e).__name__, str(e)[:200])
        break
    f = r.finding if r.status == "found" else None
    s, e = int(it["start"]), int(it["end"])
    row = {"id": it["id"], "clause": it["clause"], "label": it["risk"],
           "status": r.status, "decided_by": r.decided_by,
           "pred_risk": f.risk_level if f else None,
           "overlap": bool(f and f.citation_start < e and s < f.citation_end),
           "llm_calls": llm.calls,
           "cited_text": f.cited_text if f else None,
           "explanation": f.explanation if f else None}
    with open(OUT, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(row) + "\n")
    new += 1
    print(it["id"], it["clause"], "->", r.status, "| model calls:", llm.calls)
    time.sleep(2)
print("finished this pass")

40 labeled items, 40 already done
finished this pass


In [14]:
ABSENT_OUT = f"../eval/llm_absent_{SPLIT}.jsonl"
N_ABSENT = 8        # per clause
MAX_NEW_ABS = 20    # calls this pass. Use None for all.

idx = dev_idx if SPLIT == "dev" else test_idx
done_abs = set()
if os.path.exists(ABSENT_OUT):
    done_abs = {json.loads(l)["key"] for l in open(ABSENT_OUT, encoding="utf-8")}

rng = random.Random(7)
new = 0
for clause in TARGET_CLAUSES:
    pool = [i for i in idx if not contracts[i]["gold"][clause]]
    for i in rng.sample(pool, min(N_ABSENT, len(pool))):
        c = contracts[i]
        key = f"{clause}|{c['title']}"
        if key in done_abs:
            continue
        if MAX_NEW_ABS is not None and new >= MAX_NEW_ABS:
            break
        try:
            r = review_contract(c, hybrid, signals, thresholds, complete, clauses=[clause])[0]
        except Exception as e:
            print("stopped:", type(e).__name__, str(e)[:200])
            MAX_NEW_ABS = 0
            break
        row = {"key": key, "clause": clause, "title": c["title"],
               "status": r.status, "decided_by": r.decided_by,
               "cited_text": r.finding.cited_text if r.status == "found" else None}
        with open(ABSENT_OUT, "a", encoding="utf-8") as fh:
            fh.write(json.dumps(row) + "\n")
        new += 1
        time.sleep(2)

rows_abs = [json.loads(l) for l in open(ABSENT_OUT, encoding="utf-8")]
print(len(rows_abs), "done. statuses:", pd.Series([r["status"] for r in rows_abs]).value_counts().to_dict())

56 done. statuses: {'not_found': 52, 'found': 4}


In [17]:
rows_abs = [json.loads(l) for l in open(ABSENT_OUT, encoding="utf-8")]
print(pd.DataFrame(rows_abs).groupby(["clause", "decided_by", "status"]).size())

clause                              decided_by  status   
Cap On Liability                    gate        not_found    2
                                    llm         not_found    6
Governing Law                       gate        not_found    8
Non-Compete                         llm         found        2
                                                not_found    6
Notice Period To Terminate Renewal  llm         not_found    8
Renewal Term                        llm         not_found    8
Termination For Convenience         llm         found        1
                                                not_found    7
Uncapped Liability                  llm         found        1
                                                not_found    7
dtype: int64


In [18]:
rows_abs = [json.loads(l) for l in open(ABSENT_OUT, encoding="utf-8")]
df = pd.DataFrame(rows_abs)
df["gate"] = df.decided_by == "gate"
print(df.groupby(["clause", "decided_by", "status"]).size())
print()
for r in rows_abs:
    if r["status"] == "found":
        print(r["clause"], "|", r["title"][:60])
        print("   ", (r["cited_text"] or "")[:400])

clause                              decided_by  status   
Cap On Liability                    gate        not_found    2
                                    llm         not_found    6
Governing Law                       gate        not_found    8
Non-Compete                         llm         found        2
                                                not_found    6
Notice Period To Terminate Renewal  llm         not_found    8
Renewal Term                        llm         not_found    8
Termination For Convenience         llm         found        1
                                                not_found    7
Uncapped Liability                  llm         found        1
                                                not_found    7
dtype: int64

Termination For Convenience | VARIABLESEPARATEACCOUNT_04_30_2014-EX-13.C-UNCONDITIONAL CAP
    AIG shall have the absolute right to terminate this Agreement upon thirty (30) days' prior written notice to the Company, which notice shall